In [1]:
import numpy as np
import pandas as pd

In [2]:
csv_path = "../data/real_bitcoin_blocks_raw.csv"
df = pd.read_csv(csv_path)

df.head()

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty
0,0,2009-01-03 18:15:05+00:00,285,1,1d00ffff,0,5.000000e+09,0.0,0.0,1,1.0
1,1,2009-01-09 02:54:25+00:00,215,1,1d00ffff,1,5.000000e+09,0.0,0.0,1,1.0
2,2,2009-01-09 02:55:44+00:00,215,1,1d00ffff,2,5.000000e+09,0.0,0.0,1,1.0
3,3,2009-01-09 03:02:53+00:00,215,1,1d00ffff,3,5.000000e+09,0.0,0.0,1,1.0
4,4,2009-01-09 03:16:28+00:00,215,1,1d00ffff,4,5.000000e+09,0.0,0.0,1,1.0


In [3]:
# Either "total_output_satoshis" or "total_output_satoshis_excl_coinbase"
# Either "full" (all 9 features) or "reduced" (drop avg_transactions and avg_volume)
def preprocess(df, volume_column):
    # Rename Columns
    processed_df = df.rename(columns={
        'number': 'block_id',
        'transaction_count': 'n_transactions',
        volume_column: 'transaction_volume',
    })

    # Drop Unwanted Columns
    if volume_column == "total_output_satoshis":
        drop_columns = ['total_output_satoshis_excl_coinbase', 'total_fee_satoshis', 'block_number', 'tx_count_check', 'bits']
        
    elif volume_column == "total_output_satoshis_excl_coinbase":
        drop_columns = ['total_output_satoshis', 'total_fee_satoshis', 'block_number', 'tx_count_check', 'bits']
    
    processed_df = processed_df.drop(columns=drop_columns)

    # Convert satoshis to BTC and rename
    satoshi_columns = ['transaction_volume']
    processed_df[satoshi_columns] = processed_df[satoshi_columns] / 1e8

    # Round-robin miner assignment
    processed_df['miner_id'] = processed_df['block_id'] % 100

    # Calculate fee proxy as n_transactions * transaction_volume
    processed_df['fee_proxy'] = processed_df['n_transactions'] * processed_df['transaction_volume']
    
    # Groupby miner_id
    miner_df = processed_df.groupby('miner_id').agg(
        blocks_mined=('block_id', 'count'),
        avg_transactions=('n_transactions', 'mean'),
        avg_volume=('transaction_volume', 'mean'),
        avg_fee=('fee_proxy', 'mean'),
        fee_volatility=('fee_proxy', 'std'),
        avg_block_size=('size', 'mean'),
        difficulty=('difficulty', 'mean'),
        profitability=('fee_proxy', 'sum'),
        last_block_id=('block_id', 'max')   # last block a mined by a miner
    ).reset_index()
    
    # convert profitability from total sum to avg profitability
    miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
    miner_df['age'] = processed_df['block_id'].max() - miner_df['last_block_id']
    
    # added very small number (1 * 10^-9) to prevent division by zero
    # if any miner has avg_volume of 0
    miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
    
    # Calculate median efficiency
    median_efficiency = miner_df['efficiency'].median()
    # print(f"Median Efficiency = {median_efficiency}")
    # label miner as 1 if its efficiency is more than median, else 0
    miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)
    
    return miner_df

In [4]:
miner_df = preprocess(df, "total_output_satoshis")
miner_df.head()

,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,profitability,last_block_id,age,efficiency,label
0,0,8110,1112.876079,10848.979162,1.910743e+07,7.302996e+07,636997.443157,7.822986e+12,1.910507e+07,810900,8,0.102579,0
1,1,8110,1113.248212,10876.172794,1.882207e+07,5.546725e+07,633592.245993,7.822986e+12,1.881975e+07,810901,7,0.102357,0
2,2,8110,1120.462762,10749.633443,1.934361e+07,9.517518e+07,632985.547472,7.822986e+12,1.934123e+07,810902,6,0.104233,0
3,3,8110,1099.984957,10317.032438,1.838393e+07,5.959257e+07,631024.890999,7.822986e+12,1.838166e+07,810903,5,0.106618,0
4,4,8110,1123.225524,11427.595817,2.072525e+07,1.492980e+08,633191.847965,7.823309e+12,2.072270e+07,810904,4,0.098291,0


In [5]:
# Train and Test
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

def train_and_test(miner_df, feature_set):
    
    feature_columns = []
    if feature_set == "full":
        feature_columns = ['blocks_mined', 'avg_transactions', 'avg_volume', 
                        'avg_fee', 'fee_volatility', 'avg_block_size', 
                        'difficulty', 'profitability', 'age']
    elif feature_set == "reduced":
        feature_columns = ['blocks_mined', 
                           'avg_fee', 'fee_volatility', 'avg_block_size', 
                           'difficulty', 'profitability', 'age']

    X = miner_df[feature_columns]
    y = miner_df['label']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Scale data using normalization so that different features
    # with huge numbers and small numbers have equal importance

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = MLPClassifier(
        hidden_layer_sizes=(64, 32, 16, 8), 
        activation='relu', 
        solver='adam', 
        alpha=0.001, 
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        max_iter=100,
        batch_size=16
        )

    # print(model)

    # Fit the model (Training)
    model.fit(X_train_scaled, y_train)
    
    # Test the accuracy of the model
    # Ethan's accuracy:
    # Test Accuracy = 95%
    # 5-fold Cross Validation Accuracy = 48.75%
    y_prediction = model.predict(X_test_scaled)
    # print(f"y prediction: {y_prediction}")
    
    test_accuracy = accuracy_score(y_test, y_prediction)
    # print(test_accuracy)

    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    cv_mean = cv_scores.mean()
    # print(f"Cross-Validation Scores:     {cv_scores}")
    # print(f"Cross-Validation Mean Score: {cv_mean}")
    
    median_efficiency = miner_df['efficiency'].median()
    min_efficiency = miner_df['efficiency'].min()
    max_efficiency = miner_df['efficiency'].max()
    
    
    # High cv score found? should be closer to Ethan's Accuracy
    # corr_avg_volume_efficiency = miner_df[['avg_transactions', 'avg_volume']].corrwith(miner_df['efficiency'])
    corr_avg_transaction_efficiency = miner_df['avg_transactions'].corr(miner_df['efficiency'])
    corr_avg_volume_efficiency = miner_df['avg_volume'].corr(miner_df['efficiency'])
    
    # print(corr_avg_transaction_efficiency)
    # print(corr_avg_volume_efficiency)
    
    return dict(
        test_accuracy=test_accuracy, cv_mean=cv_mean, 
        median_efficiency=median_efficiency, min_efficiency=min_efficiency, max_efficiency=max_efficiency,
        corr_avg_transaction_efficiency=corr_avg_transaction_efficiency, corr_avg_volume_efficiency=corr_avg_volume_efficiency,
        model_iterations=model.n_iter_
        )

In [6]:
train_and_test(miner_df, "full")

{'test_accuracy': 0.8,
 'cv_mean': np.float64(0.85),
 'median_efficiency': np.float64(0.10715277569636858),
 'min_efficiency': np.float64(0.09829062403503004),
 'max_efficiency': np.float64(0.1146304128236425),
 'corr_avg_transaction_efficiency': np.float64(-0.0003131049631070735),
 'corr_avg_volume_efficiency': np.float64(-0.9829170130014887),
 'model_iterations': 25}

In [7]:
def preprocess_and_test(df, volume_column, feature_set):
    miner_df = preprocess(df, volume_column)
    results = train_and_test(miner_df, feature_set)
    full_results = {
        "feature_set": feature_set,
        "volume_column": volume_column,
        **results
    }
    
    return pd.DataFrame([full_results])

In [8]:
results_df = []

results_df.append(preprocess_and_test(df, "total_output_satoshis_excl_coinbase", "full"))
results_df.append(preprocess_and_test(df, "total_output_satoshis_excl_coinbase", "reduced"))
results_df.append(preprocess_and_test(df, "total_output_satoshis", "full"))
results_df.append(preprocess_and_test(df, "total_output_satoshis", "reduced"))

final_results = pd.concat(results_df, ignore_index=True)
final_results

,feature_set,volume_column,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,full,total_output_satoshis_excl_coinbase,0.80,0.85,0.107405,0.098501,0.114919,-0.000700,-0.982983,25
1,reduced,total_output_satoshis_excl_coinbase,0.85,0.70,0.107405,0.098501,0.114919,-0.000700,-0.982983,37
2,full,total_output_satoshis,0.80,0.85,0.107153,0.098291,0.114630,-0.000313,-0.982917,25
3,reduced,total_output_satoshis,0.85,0.70,0.107153,0.098291,0.114630,-0.000313,-0.982917,37
